In [ ]:
# ================= CURRENT FOLD LOGIC =================

# 40% train -> 24h purge -> next 20%
# (the training set is slightly less than 40% because removes the final 24 hours before validation)
# 60% train -> 24h purge -> next 20%
# 80% train -> 24h purge -> final 20%

# =============================================================================================================
# ======================================= HYPERPARAMETER TUNING ===============================================
# =============================================================================================================

import os
import sys
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import optuna

/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/MFIV_calibration/.venv01/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
sys.path.insert(0, "/Users/shah/CODE_BOOK_4/THESIS_WORKING/THESIS_2")
sys.path.insert(0, "/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/MFIV_calibration")
os.chdir("/Users/shah/CODE_BOOK_4/THESIS_WORKING/THESIS_2")

In [ ]:
# =============================================================================================================
# ======================================= COMMON PARAMETERS ===================================================
# =============================================================================================================

YEARS = [2019, 2020, 2021]
QUARTERS = ["q1", "q2", "q3", "q4"]

SEQUENCE_LENGTH = 168
FORWARD_HOURS = 24
DELTA_TARGET = 0.20

VAL_FRACTION = 0.20
TRAIN_FRACTIONS = [0.40, 0.60, 0.80]

EPOCHS = 40
PATIENCE = 6
SEED = 42

N_TRIALS = 40

In [ ]:
# =============================================================================================================
# ======================================= DATA PATHS ==========================================================
# =============================================================================================================

DATA_DIR = Path("../data/rawData")

OUTPUTS_PATH = (
    "/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/"
    "data/IS/outputs_by_year/outputs_by_year_01.pkl"
)

X_BY_YEAR_PATH = (
    "/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/"
    "data/IS/x_by_year_fut/x_by_year.pkl"
)

RFR_PATH = (
    "/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/"
    "data/IS/ready_data/d_btc_by_year_all.pkl"
)

In [ ]:
# =============================================================================================================
# ======================================= TCN IMPORTS =========================================================
# =============================================================================================================

sys.modules.pop("srcB.tcn_model_v2", None)

from srcB.tcn_model_v2 import (
    build_tcn_features,
    build_forward_targets as build_common_targets,
    make_training_sequences as make_common_sequences,
    SmallCausalTCN,
    _scalers as tcn_scalers,
    _train as train_tcn,
    _get_device as get_tcn_device,
)

In [ ]:
# =============================================================================================================
# ======================================= LOAD IS DATA ========================================================
# =============================================================================================================

def load_is_data():

    with open(OUTPUTS_PATH, "rb") as f:
        outputs_by_year = pickle.load(f)

    X_by_year = pd.read_pickle(X_BY_YEAR_PATH)

    with open(RFR_PATH, "rb") as f:
        d_btc_by_year = pickle.load(f)

    d_btc_raw_by_year = {}
    df_raw_by_year = {}

    for y in YEARS:

        d_btc_raw_by_year[y] = pd.read_parquet(
            DATA_DIR / f"btc_underlying_minutely_{y}.parquet"
        )

        df_raw_by_year[y] = pd.read_parquet(
            DATA_DIR / f"derivative_data_{y}.parquet"
        )

        d_btc_raw_by_year[y] = d_btc_raw_by_year[y].set_index(
            pd.to_datetime(d_btc_raw_by_year[y]["date_time"])
        )

        df_raw_by_year[y] = df_raw_by_year[y].set_index(
            pd.to_datetime(df_raw_by_year[y]["date_time"])
        )

        d_btc_raw_by_year[y]["rfr"] = d_btc_by_year[y]["rfr"].reindex(
            d_btc_raw_by_year[y].index,
            method="ffill"
        )

    return (
        X_by_year,
        outputs_by_year,
        d_btc_raw_by_year,
        df_raw_by_year,
    )

In [ ]:
# =============================================================================================================
# ======================================= PREPARE ONE MODEL YEAR ==============================================
# =============================================================================================================

def prepare_model_year(
    year,
    X_by_year,
    outputs_by_year,
    d_btc_raw_by_year,
    df_raw_by_year,
    build_features,
):

    X_df = X_by_year[year].copy().sort_index()

    X_df["rv"] = X_df["rt"] ** 2

    for H in [12, 24, 72, 168, 336]:
        X_df[f"MFIV_{H}"] *= H / 8760.0

    garch = pd.concat(
        [outputs_by_year[year][q]["garch"] for q in QUARTERS]
    ).sort_index()

    X_df["garch"] = garch.reindex(X_df.index)

    features = build_features(X_df)

    targets = build_common_targets(
        option_df=df_raw_by_year[year],
        time_df=d_btc_raw_by_year[year],
        times=features.dropna().index,
        forward_hours=FORWARD_HOURS,
        delta_target=DELTA_TARGET,
    )

    X, y, idx = make_common_sequences(
        features,
        targets,
        sequence_length=SEQUENCE_LENGTH,
    )

    return {
        "X": X,
        "y": y,
        "idx": idx,
        "n_features": features.shape[1],
    }

In [ ]:
# =============================================================================================================
# ======================================= PREPARE ALL MODEL YEARS =============================================
# =============================================================================================================

def prepare_model_data(
    X_by_year,
    outputs_by_year,
    d_btc_raw_by_year,
    df_raw_by_year,
    build_features,
    model_name,
):

    model_data = {}

    for y in YEARS:

        print(f"Preparing {model_name} data:", y)

        model_data[y] = prepare_model_year(
            year=y,
            X_by_year=X_by_year,
            outputs_by_year=outputs_by_year,
            d_btc_raw_by_year=d_btc_raw_by_year,
            df_raw_by_year=df_raw_by_year,
            build_features=build_features,
        )

        print("Sequences:", len(model_data[y]["idx"]))

    return model_data

# =============================================================================================================
# ======================================= ROLLING CHRONOLOGICAL FOLD ==========================================
# =============================================================================================================

def make_rolling_fold(data, train_fraction):

    X = data["X"]
    y = data["y"]
    idx = data["idx"]

    val_start_i = int(len(idx) * train_fraction)
    val_end_i = int(len(idx) * (train_fraction + VAL_FRACTION))

    val_start = idx[val_start_i]

    train_mask = (
        idx
        < val_start - pd.Timedelta(hours=FORWARD_HOURS)
    )

    if val_end_i < len(idx):

        val_end = idx[val_end_i]

        val_mask = (
            (idx >= val_start)
            & (idx < val_end)
        )

    else:
        val_mask = idx >= val_start

    return {
        "X_train": X[train_mask],
        "y_train": y[train_mask],
        "X_val": X[val_mask],
        "y_val": y[val_mask],
    }

# =============================================================================================================
# ======================================= FIT ONE MODEL FOLD ==================================================
# =============================================================================================================

def fit_model_fold(
    fold,
    model,
    params,
    scalers,
    train,
    device,
    seed,
):

    np.random.seed(seed)
    torch.manual_seed(seed)

    X_train = fold["X_train"]
    y_train = fold["y_train"]

    X_val = fold["X_val"]
    y_val = fold["y_val"]

    x_mean, x_std, y_mean, y_std = scalers(
        X_train,
        y_train
    )

    X_train = ((X_train - x_mean) / x_std).astype(np.float32)
    X_val = ((X_val - x_mean) / x_std).astype(np.float32)

    y_train = ((y_train - y_mean) / y_std).astype(np.float32)
    y_val = ((y_val - y_mean) / y_std).astype(np.float32)

    model, best_epoch, history = train(
        model=model,
        X=X_train,
        y=y_train,
        device=device,
        epochs=EPOCHS,
        batch_size=params["batch_size"],
        learning_rate=params["learning_rate"],
        weight_decay=params["weight_decay"],
        X_val=X_val,
        y_val=y_val,
        patience=PATIENCE,
    )

    return history["val_loss"].min()


# =============================================================================================================
# ======================================= TCN SEARCH SPACE ====================================================
# =============================================================================================================

def suggest_tcn_params(trial):

    channel_width = trial.suggest_categorical(
        "channel_width",
        [8, 16, 32]
    )

    num_blocks = trial.suggest_categorical(
        "num_blocks",
        [5, 6, 7]
    )

    kernel_size = trial.suggest_categorical(
        "kernel_size",
        [3, 5]
    )

    dropout = trial.suggest_categorical(
        "dropout",
        [0.0, 0.10, 0.20]
    )

    learning_rate = trial.suggest_float(
        "learning_rate",
        1e-4,
        3e-3,
        log=True
    )

    weight_decay = trial.suggest_categorical(
        "weight_decay",
        [0.0, 1e-6, 1e-5, 1e-4, 1e-3]
    )

    batch_size = trial.suggest_categorical(
        "batch_size",
        [64, 128, 256]
    )

    channels = (channel_width,) * num_blocks

    return {
        "channel_width": channel_width,
        "num_blocks": num_blocks,
        "channels": channels,
        "kernel_size": kernel_size,
        "dropout": dropout,
        "learning_rate": learning_rate,
        "weight_decay": weight_decay,
        "batch_size": batch_size,
    }


# =============================================================================================================
# ======================================= CHECK TCN RECEPTIVE FIELD ===========================================
# =============================================================================================================

def valid_tcn_receptive_field(params):

    receptive_field = (
        1
        + 2
        * (params["kernel_size"] - 1)
        * (2 ** params["num_blocks"] - 1)
    )

    return receptive_field >= SEQUENCE_LENGTH

In [ ]:
# =============================================================================================================
# ======================================= EVALUATE ONE TCN TRIAL ==============================================
# =============================================================================================================

def evaluate_tcn_trial(trial, tcn_data):

    params = suggest_tcn_params(trial)

    if not valid_tcn_receptive_field(params):
        raise optuna.TrialPruned()

    all_losses = []

    resources = [1, 2, 4]

    for resource, train_fraction in zip(resources, TRAIN_FRACTIONS):

        round_losses = []

        for y in YEARS:

            fold = make_rolling_fold(
                tcn_data[y],
                train_fraction
            )
            np.random.seed(SEED)
            torch.manual_seed(SEED)
            model = SmallCausalTCN(
                n_features=tcn_data[y]["n_features"],
                channels=params["channels"],
                kernel_size=params["kernel_size"],
                dropout=params["dropout"],
            )

            val_mse = fit_model_fold(
                fold=fold,
                model=model,
                params=params,
                scalers=tcn_scalers,
                train=train_tcn,
                device=get_tcn_device(),
                seed=SEED,
            )
            round_losses.append(val_mse)

        all_losses.extend(round_losses)

        mean_val_mse = float(np.mean(all_losses))

        trial.report(
            mean_val_mse,
            step=resource
        )

        if trial.should_prune():
            raise optuna.TrialPruned()

    return mean_val_mse


# =============================================================================================================
# ======================================= RUN TCN TUNING ======================================================
# =============================================================================================================

def tune_tcn(tcn_data):

    sampler = optuna.samplers.TPESampler(
        seed=SEED,
        n_startup_trials=10
    )

    pruner = optuna.pruners.SuccessiveHalvingPruner(
        min_resource=1,
        reduction_factor=2
    )

    study = optuna.create_study(
        direction="minimize",
        sampler=sampler,
        pruner=pruner
    )

    study.optimize(
        lambda trial: evaluate_tcn_trial(
            trial,
            tcn_data
        ),
        n_trials=N_TRIALS
    )

    return study

In [ ]:
# =============================================================================================================
# ======================================= MAIN ================================================================
# =============================================================================================================


(
    X_by_year,
    outputs_by_year,
    d_btc_raw_by_year,
    df_raw_by_year,
) = load_is_data()

tcn_data = prepare_model_data(
    X_by_year=X_by_year,
    outputs_by_year=outputs_by_year,
    d_btc_raw_by_year=d_btc_raw_by_year,
    df_raw_by_year=df_raw_by_year,
    build_features=build_tcn_features,
    model_name="TCN",
)

tcn_study = tune_tcn(tcn_data)

print("\nBEST TCN VALIDATION MSE")
print(tcn_study.best_value)

print("\nBEST TCN PARAMETERS")
print(tcn_study.best_params)

best_tcn_params = tcn_study.best_params.copy()

best_tcn_params["channels"] = (
    best_tcn_params["channel_width"],
) * best_tcn_params["num_blocks"]

print("\nTCN CHANNELS")
print(best_tcn_params["channels"])

Preparing TCN data: 2019
Sequences: 2194
Preparing TCN data: 2020
Sequences: 4093
Preparing TCN data: 2021


[I 2026-09-03 00:47:25,095] A new study created in memory with name: no-name-abafe946-8950-4dae-806d-44203befb9ed


Sequences: 5008


[I 2026-09-03 00:47:50,099] Trial 0 finished with value: 0.6748935977617899 and parameters: {'channel_width': 16, 'num_blocks': 5, 'kernel_size': 5, 'dropout': 0.1, 'learning_rate': 0.002708160864249967, 'weight_decay': 0.0, 'batch_size': 64}. Best is trial 0 with value: 0.6748935977617899.
[I 2026-09-03 00:48:26,513] Trial 1 pruned. 
[I 2026-09-03 00:48:55,225] Trial 2 finished with value: 0.6671932985385259 and parameters: {'channel_width': 16, 'num_blocks': 5, 'kernel_size': 5, 'dropout': 0.2, 'learning_rate': 0.0013962563737015762, 'weight_decay': 0.0, 'batch_size': 256}. Best is trial 2 with value: 0.6671932985385259.
[I 2026-09-03 00:49:38,673] Trial 3 pruned. 
[I 2026-09-03 00:50:11,939] Trial 4 pruned. 
[I 2026-09-03 00:50:21,245] Trial 5 pruned. 
[I 2026-09-03 00:50:37,158] Trial 6 pruned. 
[I 2026-09-03 00:50:37,159] Trial 7 pruned. 
[I 2026-09-03 00:51:48,800] Trial 8 pruned. 
[I 2026-09-03 00:51:48,806] Trial 9 pruned. 
[I 2026-09-03 00:52:37,686] Trial 10 finished with val


BEST TCN VALIDATION MSE
0.6645874016814761

BEST TCN PARAMETERS
{'channel_width': 16, 'num_blocks': 6, 'kernel_size': 5, 'dropout': 0.2, 'learning_rate': 0.0020250713194581147, 'weight_decay': 0.0001, 'batch_size': 256}

TCN CHANNELS
(16, 16, 16, 16, 16, 16)


In [ ]:
# =============================================================================================================
# ======================================= SAVE TCN HPO RESULTS ================================================
# =============================================================================================================
    
HYPERPARAM_DIR = "/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/data/IS/hyperparameters"

os.makedirs(HYPERPARAM_DIR, exist_ok=True)

with open(f"{HYPERPARAM_DIR}/tcn_study.pkl", "wb") as f:
    pickle.dump(tcn_study, f)

with open(f"{HYPERPARAM_DIR}/tcn_best_params.pkl", "wb") as f:
    pickle.dump(best_tcn_params, f)

In [ ]:
# =============================================================================================================
# ======================================= GRU IMPORTS =========================================================
# =============================================================================================================

sys.modules.pop("srcB.gru_model", None)

from srcB.gru_model import (
    build_gru_features,
    SmallGRU,
    _scalers as gru_scalers,
    _train as train_gru,
    _get_device as get_gru_device,
)

GRU_SEED = 11

In [ ]:
# =============================================================================================================
# ======================================= GRU SEARCH SPACE ====================================================
# =============================================================================================================

def suggest_gru_params(trial):

    hidden_size = trial.suggest_categorical(
        "hidden_size",
        [16, 32, 64]
    )

    num_layers = trial.suggest_categorical(
        "num_layers",
        [1, 2, 3, 4, 6]
    )
    if num_layers == 1:
        dropout = 0.0
    else:
        dropout = trial.suggest_categorical(
            "dropout",
            [0.0, 0.10, 0.20, 0.30]
        )
    learning_rate = trial.suggest_float(
        "learning_rate",
        1e-4,
        3e-3,
        log=True
    )

    weight_decay = trial.suggest_categorical(
        "weight_decay",
        [0.0, 1e-6, 1e-5, 1e-4, 1e-3]
    )

    batch_size = trial.suggest_categorical(
        "batch_size",
        [64, 128, 256]
    )

    return {
        "hidden_size": hidden_size,
        "num_layers": num_layers,
        "dropout": dropout,
        "learning_rate": learning_rate,
        "weight_decay": weight_decay,
        "batch_size": batch_size,
    }

In [ ]:
# =============================================================================================================
# ======================================= EVALUATE ONE GRU TRIAL ==============================================
# =============================================================================================================

def evaluate_gru_trial(trial, gru_data):

    params = suggest_gru_params(trial)

    all_losses = []

    resources = [1, 2, 4]

    for resource, train_fraction in zip(resources, TRAIN_FRACTIONS):

        round_losses = []

        for y in YEARS:

            fold = make_rolling_fold(
                gru_data[y],
                train_fraction
            )

            np.random.seed(GRU_SEED)
            torch.manual_seed(GRU_SEED)

            model = SmallGRU(
                n_features=gru_data[y]["n_features"],
                hidden_size=params["hidden_size"],
                num_layers=params["num_layers"],
                dropout=params["dropout"],
            )

            val_mse = fit_model_fold(
                fold=fold,
                model=model,
                params=params,
                scalers=gru_scalers,
                train=train_gru,
                device=get_gru_device(),
                seed=GRU_SEED,
            )

            round_losses.append(val_mse)

        all_losses.extend(round_losses)

        mean_val_mse = float(np.mean(all_losses))

        trial.report(
            mean_val_mse,
            step=resource
        )

        if trial.should_prune():
            raise optuna.TrialPruned()

    return mean_val_mse



# =============================================================================================================
# ======================================= RUN GRU TUNING ======================================================
# =============================================================================================================

def tune_gru(gru_data):

    sampler = optuna.samplers.TPESampler(
        seed=GRU_SEED,
        n_startup_trials=10
    )

    pruner = optuna.pruners.SuccessiveHalvingPruner(
        min_resource=1,
        reduction_factor=2
    )

    study = optuna.create_study(
        direction="minimize",
        sampler=sampler,
        pruner=pruner
    )

    study.optimize(
        lambda trial: evaluate_gru_trial(
            trial,
            gru_data
        ),
        n_trials=N_TRIALS
    )

    return study

In [ ]:
# =============================================================================================================
# ======================================= GRU MAIN ============================================================
# =============================================================================================================

(
    X_by_year,
    outputs_by_year,
    d_btc_raw_by_year,
    df_raw_by_year,
) = load_is_data()

gru_data = prepare_model_data(
    X_by_year=X_by_year,
    outputs_by_year=outputs_by_year,
    d_btc_raw_by_year=d_btc_raw_by_year,
    df_raw_by_year=df_raw_by_year,
    build_features=build_gru_features,
    model_name="GRU",
)

gru_study = tune_gru(gru_data)

print("\nBEST GRU VALIDATION MSE")
print(gru_study.best_value)

print("\nBEST GRU PARAMETERS")
print(gru_study.best_params)

best_gru_params = gru_study.best_params.copy()
if best_gru_params["num_layers"] == 1:
    best_gru_params["dropout"] = 0.0

Preparing GRU data: 2019
Sequences: 2194
Preparing GRU data: 2020
Sequences: 4093
Preparing GRU data: 2021


[I 2026-09-03 01:06:04,070] A new study created in memory with name: no-name-67817331-522c-4724-abd7-0425a258b215


Sequences: 5008


[I 2026-09-03 01:07:44,022] Trial 0 finished with value: 0.6722007327609592 and parameters: {'hidden_size': 64, 'num_layers': 1, 'learning_rate': 0.0024612890751570505, 'weight_decay': 0.0001, 'batch_size': 128}. Best is trial 0 with value: 0.6722007327609592.
[I 2026-09-03 01:11:54,982] Trial 1 finished with value: 0.6650917712185118 and parameters: {'hidden_size': 32, 'num_layers': 2, 'dropout': 0.1, 'learning_rate': 0.0005113505915529021, 'weight_decay': 1e-06, 'batch_size': 128}. Best is trial 1 with value: 0.6650917712185118.
[I 2026-09-03 01:12:52,701] Trial 2 finished with value: 0.6593970722622342 and parameters: {'hidden_size': 32, 'num_layers': 1, 'learning_rate': 0.001469180524875077, 'weight_decay': 0.0, 'batch_size': 256}. Best is trial 2 with value: 0.6593970722622342.
[I 2026-09-03 01:22:23,372] Trial 3 finished with value: 0.6632340517308977 and parameters: {'hidden_size': 64, 'num_layers': 4, 'dropout': 0.1, 'learning_rate': 0.0015772333927311457, 'weight_decay': 1e-06


BEST GRU VALIDATION MSE
0.6543862173954645

BEST GRU PARAMETERS
{'hidden_size': 32, 'num_layers': 6, 'dropout': 0.0, 'learning_rate': 0.00045857703103654784, 'weight_decay': 1e-06, 'batch_size': 256}


In [ ]:
# =============================================================================================================
# ======================================= SAVE GRU HPO RESULTS ================================================
# =============================================================================================================

HYPERPARAM_DIR = "/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/data/IS/hyperparameters"

os.makedirs(HYPERPARAM_DIR, exist_ok=True)

with open(f"{HYPERPARAM_DIR}/gru_study.pkl", "wb") as f:
    pickle.dump(gru_study, f)

with open(f"{HYPERPARAM_DIR}/gru_best_params.pkl", "wb") as f:
    pickle.dump(best_gru_params, f)

In [ ]:
# =============================================================================================================
# ======================================= TSM IMPORTS =========================================================
# =============================================================================================================

sys.modules.pop("srcB.tsm_model", None)

from srcB.tsm_model import (
    build_tsm_features,
    SmallTSMixer,
    _scalers as tsm_scalers,
    _train as train_tsm,
    _get_device as get_tsm_device,
)

TSM_SEED = 42

In [ ]:
# =============================================================================================================
# ======================================= TSM SEARCH SPACE ====================================================
# =============================================================================================================

def suggest_tsm_params(trial):

    num_blocks = trial.suggest_categorical(
        "num_blocks",
        [1, 2, 3, 4]
    )

    feature_hidden = trial.suggest_categorical(
        "feature_hidden",
        [32, 64, 128]
    )

    dropout = trial.suggest_categorical(
        "dropout",
        [0.0, 0.10, 0.20]
    )

    learning_rate = trial.suggest_float(
        "learning_rate",
        1e-4,
        3e-3,
        log=True
    )

    weight_decay = trial.suggest_categorical(
        "weight_decay",
        [0.0, 1e-6, 1e-5, 1e-4, 1e-3]
    )

    batch_size = trial.suggest_categorical(
        "batch_size",
        [64, 128, 256]
    )

    return {
        "num_blocks": num_blocks,
        "feature_hidden": feature_hidden,
        "dropout": dropout,
        "learning_rate": learning_rate,
        "weight_decay": weight_decay,
        "batch_size": batch_size,
    }

In [ ]:
# =============================================================================================================
# ======================================= EVALUATE ONE TSM TRIAL ==============================================
# =============================================================================================================

def evaluate_tsm_trial(trial, tsm_data):

    params = suggest_tsm_params(trial)

    all_losses = []

    resources = [1, 2, 4]

    for resource, train_fraction in zip(resources, TRAIN_FRACTIONS):

        round_losses = []

        for y in YEARS:

            fold = make_rolling_fold(
                tsm_data[y],
                train_fraction
            )

            np.random.seed(TSM_SEED)
            torch.manual_seed(TSM_SEED)

            model = SmallTSMixer(
                n_features=tsm_data[y]["n_features"],
                sequence_length=SEQUENCE_LENGTH,
                num_blocks=params["num_blocks"],
                feature_hidden=params["feature_hidden"],
                dropout=params["dropout"],
            )

            val_mse = fit_model_fold(
                fold=fold,
                model=model,
                params=params,
                scalers=tsm_scalers,
                train=train_tsm,
                device=get_tsm_device(),
                seed=TSM_SEED,
            )

            round_losses.append(val_mse)

        all_losses.extend(round_losses)

        mean_val_mse = float(np.mean(all_losses))

        trial.report(
            mean_val_mse,
            step=resource
        )

        if trial.should_prune():
            raise optuna.TrialPruned()

    return mean_val_mse


# =============================================================================================================
# ======================================= RUN TSM TUNING ======================================================
# =============================================================================================================

def tune_tsm(tsm_data):

    sampler = optuna.samplers.TPESampler(
        seed=TSM_SEED,
        n_startup_trials=10
    )

    pruner = optuna.pruners.SuccessiveHalvingPruner(
        min_resource=1,
        reduction_factor=2
    )

    study = optuna.create_study(
        direction="minimize",
        sampler=sampler,
        pruner=pruner
    )

    study.optimize(
        lambda trial: evaluate_tsm_trial(
            trial,
            tsm_data
        ),
        n_trials=N_TRIALS
    )

    return study

In [ ]:
# =============================================================================================================
# ======================================= TSM MAIN ============================================================
# =============================================================================================================

(
    X_by_year,
    outputs_by_year,
    d_btc_raw_by_year,
    df_raw_by_year,
) = load_is_data()

tsm_data = prepare_model_data(
    X_by_year=X_by_year,
    outputs_by_year=outputs_by_year,
    d_btc_raw_by_year=d_btc_raw_by_year,
    df_raw_by_year=df_raw_by_year,
    build_features=build_tsm_features,
    model_name="TSM",
)

tsm_study = tune_tsm(tsm_data)

print("\nBEST TSM VALIDATION MSE")
print(tsm_study.best_value)

print("\nBEST TSM PARAMETERS")
print(tsm_study.best_params)

best_tsm_params = tsm_study.best_params.copy()

Preparing TSM data: 2019
Sequences: 2194
Preparing TSM data: 2020
Sequences: 4093
Preparing TSM data: 2021


[I 2026-09-03 05:43:42,275] A new study created in memory with name: no-name-8bc7031c-f4f8-4a45-b0e8-69d0c7373a05


Sequences: 5008


[I 2026-09-03 05:44:11,224] Trial 0 finished with value: 0.6881322215000788 and parameters: {'num_blocks': 2, 'feature_hidden': 32, 'dropout': 0.0, 'learning_rate': 0.00010725209743172001, 'weight_decay': 0.0, 'batch_size': 128}. Best is trial 0 with value: 0.6881322215000788.
[I 2026-09-03 05:44:26,017] Trial 1 pruned. 
[I 2026-09-03 05:44:31,746] Trial 2 pruned. 
[I 2026-09-03 05:44:39,886] Trial 3 pruned. 
[I 2026-09-03 05:44:42,322] Trial 4 pruned. 
[I 2026-09-03 05:45:05,024] Trial 5 pruned. 
[I 2026-09-03 05:45:28,192] Trial 6 finished with value: 0.68617359134886 and parameters: {'num_blocks': 2, 'feature_hidden': 32, 'dropout': 0.0, 'learning_rate': 0.00021711402231880007, 'weight_decay': 1e-05, 'batch_size': 64}. Best is trial 6 with value: 0.68617359134886.
[I 2026-09-03 05:45:35,293] Trial 7 pruned. 
[I 2026-09-03 05:45:40,799] Trial 8 pruned. 
[I 2026-09-03 05:45:47,975] Trial 9 pruned. 
[I 2026-09-03 05:46:00,983] Trial 10 pruned. 
[I 2026-09-03 05:46:16,857] Trial 11 prun


BEST TSM VALIDATION MSE
0.6715962274207009

BEST TSM PARAMETERS
{'num_blocks': 2, 'feature_hidden': 64, 'dropout': 0.2, 'learning_rate': 0.0015691128161256886, 'weight_decay': 0.0001, 'batch_size': 256}


In [ ]:
# =============================================================================================================
# ======================================= SAVE TSM HPO RESULTS ================================================
# =============================================================================================================

HYPERPARAM_DIR = "/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/data/IS/hyperparameters"

os.makedirs(HYPERPARAM_DIR, exist_ok=True)

with open(f"{HYPERPARAM_DIR}/tsm_study.pkl", "wb") as f:
    pickle.dump(tsm_study, f)

with open(f"{HYPERPARAM_DIR}/tsm_best_params.pkl", "wb") as f:
    pickle.dump(best_tsm_params, f)

In [ ]:
# =============================================================================================================
# ======================================= ITRANSFORMER IMPORTS =================================================
# =============================================================================================================

sys.modules.pop("srcB.itr_model", None)

from srcB.itr_model import (
    build_itr_features,
    SmallITransformer,
    _scalers as itr_scalers,
    _train as train_itr,
    _get_device as get_itr_device,
)

ITR_SEED = 42

In [ ]:
# =============================================================================================================
# ======================================= ITRANSFORMER SEARCH SPACE ===========================================
# =============================================================================================================

def suggest_itr_params(trial):

    d_model = trial.suggest_categorical(
        "d_model",
        [32, 64, 128]
    )

    n_heads = trial.suggest_categorical(
        "n_heads",
        [2, 4, 8]
    )

    e_layers = trial.suggest_categorical(
        "e_layers",
        [1, 2, 3]
    )

    d_ff_ratio = trial.suggest_categorical(
        "d_ff_ratio",
        [2, 4]
    )

    dropout = trial.suggest_categorical(
        "dropout",
        [0.0, 0.10, 0.20]
    )

    learning_rate = trial.suggest_float(
        "learning_rate",
        1e-4,
        3e-3,
        log=True
    )

    weight_decay = trial.suggest_categorical(
        "weight_decay",
        [0.0, 1e-6, 1e-5, 1e-4, 1e-3]
    )

    batch_size = trial.suggest_categorical(
        "batch_size",
        [64, 128, 256]
    )

    d_ff = d_model * d_ff_ratio

    return {
        "d_model": d_model,
        "n_heads": n_heads,
        "e_layers": e_layers,
        "d_ff_ratio": d_ff_ratio,
        "d_ff": d_ff,
        "dropout": dropout,
        "learning_rate": learning_rate,
        "weight_decay": weight_decay,
        "batch_size": batch_size,
    }

In [ ]:
# =============================================================================================================
# ======================================= EVALUATE ONE ITRANSFORMER TRIAL =====================================
# =============================================================================================================

def evaluate_itr_trial(trial, itr_data):

    params = suggest_itr_params(trial)

    all_losses = []

    resources = [1, 2, 4]

    for resource, train_fraction in zip(resources, TRAIN_FRACTIONS):

        round_losses = []

        for y in YEARS:

            fold = make_rolling_fold(
                itr_data[y],
                train_fraction
            )

            np.random.seed(ITR_SEED)
            torch.manual_seed(ITR_SEED)

            model = SmallITransformer(
                n_features=itr_data[y]["n_features"],
                sequence_length=SEQUENCE_LENGTH,
                d_model=params["d_model"],
                n_heads=params["n_heads"],
                e_layers=params["e_layers"],
                d_ff=params["d_ff"],
                dropout=params["dropout"],
            )

            val_mse = fit_model_fold(
                fold=fold,
                model=model,
                params=params,
                scalers=itr_scalers,
                train=train_itr,
                device=get_itr_device(),
                seed=ITR_SEED,
            )

            round_losses.append(val_mse)

        all_losses.extend(round_losses)

        mean_val_mse = float(np.mean(all_losses))

        trial.report(
            mean_val_mse,
            step=resource
        )

        if trial.should_prune():
            raise optuna.TrialPruned()

    return mean_val_mse

In [ ]:
# =============================================================================================================
# ======================================= RUN ITRANSFORMER TUNING =============================================
# =============================================================================================================

def tune_itr(itr_data):

    sampler = optuna.samplers.TPESampler(
        seed=ITR_SEED,
        n_startup_trials=10
    )

    pruner = optuna.pruners.SuccessiveHalvingPruner(
        min_resource=1,
        reduction_factor=2
    )

    study = optuna.create_study(
        direction="minimize",
        sampler=sampler,
        pruner=pruner
    )

    study.optimize(
        lambda trial: evaluate_itr_trial(
            trial,
            itr_data
        ),
        n_trials=N_TRIALS
    )

    return study

In [ ]:
# =============================================================================================================
# ======================================= ITRANSFORMER MAIN ===================================================
# =============================================================================================================

(
    X_by_year,
    outputs_by_year,
    d_btc_raw_by_year,
    df_raw_by_year,
) = load_is_data()

itr_data = prepare_model_data(
    X_by_year=X_by_year,
    outputs_by_year=outputs_by_year,
    d_btc_raw_by_year=d_btc_raw_by_year,
    df_raw_by_year=df_raw_by_year,
    build_features=build_itr_features,
    model_name="ITRANSFORMER",
)

itr_study = tune_itr(itr_data)

print("\nBEST ITRANSFORMER VALIDATION MSE")
print(itr_study.best_value)

print("\nBEST ITRANSFORMER PARAMETERS")
print(itr_study.best_params)

best_itr_params = itr_study.best_params.copy()

best_itr_params["d_ff"] = (
    best_itr_params["d_model"]
    * best_itr_params["d_ff_ratio"]
)

print("\nITR D_FF")
print(best_itr_params["d_ff"])

Preparing ITRANSFORMER data: 2019
Sequences: 2194
Preparing ITRANSFORMER data: 2020
Sequences: 4093
Preparing ITRANSFORMER data: 2021


[I 2026-09-03 05:57:26,394] A new study created in memory with name: no-name-c7fbbc31-bb8c-4d1d-a6b7-5d6253a2e9d1


Sequences: 5008


[I 2026-09-03 05:57:50,599] Trial 0 finished with value: 0.734983868069119 and parameters: {'d_model': 64, 'n_heads': 2, 'e_layers': 2, 'd_ff_ratio': 2, 'dropout': 0.0, 'learning_rate': 0.00018559980846490597, 'weight_decay': 1e-05, 'batch_size': 64}. Best is trial 0 with value: 0.734983868069119.
[I 2026-09-03 05:57:57,130] Trial 1 pruned. 
[I 2026-09-03 05:58:00,585] Trial 2 pruned. 
[I 2026-09-03 05:58:16,911] Trial 3 finished with value: 0.7181734459267722 and parameters: {'d_model': 32, 'n_heads': 4, 'e_layers': 2, 'd_ff_ratio': 2, 'dropout': 0.0, 'learning_rate': 0.00012413189635294234, 'weight_decay': 0.001, 'batch_size': 256}. Best is trial 3 with value: 0.7181734459267722.
[I 2026-09-03 05:58:27,063] Trial 4 pruned. 
[I 2026-09-03 05:58:31,951] Trial 5 pruned. 
[I 2026-09-03 05:58:52,716] Trial 6 pruned. 
[I 2026-09-03 05:59:13,211] Trial 7 pruned. 
[I 2026-09-03 05:59:21,879] Trial 8 pruned. 
[I 2026-09-03 05:59:30,112] Trial 9 pruned. 
[I 2026-09-03 05:59:35,520] Trial 10 pr


BEST ITRANSFORMER VALIDATION MSE
0.7050159590111839

BEST ITRANSFORMER PARAMETERS
{'d_model': 32, 'n_heads': 4, 'e_layers': 3, 'd_ff_ratio': 4, 'dropout': 0.2, 'learning_rate': 0.0007763377195518329, 'weight_decay': 0.0, 'batch_size': 256}

ITR D_FF
128


In [ ]:
# =============================================================================================================
# ======================================= SAVE ITRANSFORMER HPO RESULTS =======================================
# =============================================================================================================

HYPERPARAM_DIR = "/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/data/IS/hyperparameters"

os.makedirs(HYPERPARAM_DIR, exist_ok=True)

with open(f"{HYPERPARAM_DIR}/itr_study.pkl", "wb") as f:
    pickle.dump(itr_study, f)

with open(f"{HYPERPARAM_DIR}/itr_best_params.pkl", "wb") as f:
    pickle.dump(best_itr_params, f)

In [ ]:
# =============================================================================================================
# ======================================= DLINEAR IMPORTS =====================================================
# =============================================================================================================

sys.modules.pop("srcB.dl_model", None)

from srcB.dl_model import (
    build_dl_features,
    SmallDLinear,
    _scalers as dl_scalers,
    _train as train_dl,
    _get_device as get_dl_device,
)

DL_SEED = 42

In [ ]:
# =============================================================================================================
# ======================================= DLINEAR SEARCH SPACE ================================================
# =============================================================================================================

def suggest_dl_params(trial):

    moving_avg = trial.suggest_categorical(
        "moving_avg",
        [13, 25, 49, 73]
    )

    individual = trial.suggest_categorical(
        "individual",
        [False, True]
    )

    learning_rate = trial.suggest_float(
        "learning_rate",
        1e-4,
        3e-3,
        log=True
    )

    weight_decay = trial.suggest_categorical(
        "weight_decay",
        [0.0, 1e-6, 1e-5, 1e-4, 1e-3]
    )

    batch_size = trial.suggest_categorical(
        "batch_size",
        [64, 128, 256]
    )

    return {
        "moving_avg": moving_avg,
        "individual": individual,
        "learning_rate": learning_rate,
        "weight_decay": weight_decay,
        "batch_size": batch_size,
    }
    
    
# =============================================================================================================
# ======================================= EVALUATE ONE DLINEAR TRIAL ==========================================
# =============================================================================================================

def evaluate_dl_trial(trial, dl_data):

    params = suggest_dl_params(trial)

    all_losses = []

    resources = [1, 2, 4]

    for resource, train_fraction in zip(resources, TRAIN_FRACTIONS):

        round_losses = []

        for y in YEARS:

            fold = make_rolling_fold(
                dl_data[y],
                train_fraction
            )

            np.random.seed(DL_SEED)
            torch.manual_seed(DL_SEED)

            model = SmallDLinear(
                n_features=dl_data[y]["n_features"],
                sequence_length=SEQUENCE_LENGTH,
                moving_avg=params["moving_avg"],
                individual=params["individual"],
            )

            val_mse = fit_model_fold(
                fold=fold,
                model=model,
                params=params,
                scalers=dl_scalers,
                train=train_dl,
                device=get_dl_device(),
                seed=DL_SEED,
            )

            round_losses.append(val_mse)

        all_losses.extend(round_losses)

        mean_val_mse = float(np.mean(all_losses))

        trial.report(
            mean_val_mse,
            step=resource
        )

        if trial.should_prune():
            raise optuna.TrialPruned()

    return mean_val_mse


# =============================================================================================================
# ======================================= RUN DLINEAR TUNING ==================================================
# =============================================================================================================

def tune_dl(dl_data):

    sampler = optuna.samplers.TPESampler(
        seed=DL_SEED,
        n_startup_trials=10
    )

    pruner = optuna.pruners.SuccessiveHalvingPruner(
        min_resource=1,
        reduction_factor=2
    )

    study = optuna.create_study(
        direction="minimize",
        sampler=sampler,
        pruner=pruner
    )

    study.optimize(
        lambda trial: evaluate_dl_trial(
            trial,
            dl_data
        ),
        n_trials=N_TRIALS
    )

    return study

In [ ]:
# =============================================================================================================
# ======================================= DLINEAR MAIN ========================================================
# =============================================================================================================

(
    X_by_year,
    outputs_by_year,
    d_btc_raw_by_year,
    df_raw_by_year,
) = load_is_data()

dl_data = prepare_model_data(
    X_by_year=X_by_year,
    outputs_by_year=outputs_by_year,
    d_btc_raw_by_year=d_btc_raw_by_year,
    df_raw_by_year=df_raw_by_year,
    build_features=build_dl_features,
    model_name="DLINEAR",
)

dl_study = tune_dl(dl_data)

print("\nBEST DLINEAR VALIDATION MSE")
print(dl_study.best_value)

print("\nBEST DLINEAR PARAMETERS")
print(dl_study.best_params)

best_dl_params = dl_study.best_params.copy()

Preparing DLINEAR data: 2019
Sequences: 2194
Preparing DLINEAR data: 2020
Sequences: 4093
Preparing DLINEAR data: 2021


[I 2026-09-03 06:09:11,731] A new study created in memory with name: no-name-dd0408e3-b55b-49e0-8dd8-003b2a70ad23


Sequences: 5008


[I 2026-09-03 06:09:45,190] Trial 0 finished with value: 0.6965105384588242 and parameters: {'moving_avg': 25, 'individual': False, 'learning_rate': 0.00012184186502221769, 'weight_decay': 0.001, 'batch_size': 64}. Best is trial 0 with value: 0.6965105384588242.
[I 2026-09-03 06:09:55,783] Trial 1 pruned. 
[I 2026-09-03 06:09:58,118] Trial 2 pruned. 
[I 2026-09-03 06:10:00,598] Trial 3 pruned. 
[I 2026-09-03 06:10:10,175] Trial 4 pruned. 
[I 2026-09-03 06:10:14,304] Trial 5 pruned. 
[I 2026-09-03 06:10:24,722] Trial 6 finished with value: 0.6924370908074908 and parameters: {'moving_avg': 49, 'individual': False, 'learning_rate': 0.0005917520523090664, 'weight_decay': 0.001, 'batch_size': 256}. Best is trial 6 with value: 0.6924370908074908.
[I 2026-09-03 06:10:33,181] Trial 7 pruned. 
[I 2026-09-03 06:10:39,295] Trial 8 pruned. 
[I 2026-09-03 06:10:50,991] Trial 9 pruned. 
[I 2026-09-03 06:10:55,628] Trial 10 pruned. 
[I 2026-09-03 06:11:31,835] Trial 11 pruned. 
[I 2026-09-03 06:11:47


BEST DLINEAR VALIDATION MSE
0.6783654540777206

BEST DLINEAR PARAMETERS
{'moving_avg': 13, 'individual': False, 'learning_rate': 0.002992237749481038, 'weight_decay': 0.0001, 'batch_size': 256}


In [ ]:
# =============================================================================================================
# ======================================= SAVE DLINEAR HPO RESULTS ============================================
# =============================================================================================================

HYPERPARAM_DIR = "/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/data/IS/hyperparameters"

os.makedirs(HYPERPARAM_DIR, exist_ok=True)

with open(f"{HYPERPARAM_DIR}/dl_study.pkl", "wb") as f:
    pickle.dump(dl_study, f)

with open(f"{HYPERPARAM_DIR}/dl_best_params.pkl", "wb") as f:
    pickle.dump(best_dl_params, f)

In [ ]:
tcn_study

In [ ]:
tcn_study.items()

AttributeError: 'Study' object has no attribute 'items'

In [ ]:
type(tcn_study)

optuna.study.study.Study

In [ ]:
tcn_study.best_value

0.6645874016814761

In [ ]:
tcn_study.best_params

{'channel_width': 16,
 'num_blocks': 6,
 'kernel_size': 5,
 'dropout': 0.2,
 'learning_rate': 0.0020250713194581147,
 'weight_decay': 0.0001,
 'batch_size': 256}

In [ ]:
tcn_study.trials

[FrozenTrial(number=0, state=<TrialState.COMPLETE: 1>, values=[0.6748935977617899], datetime_start=datetime.datetime(2026, 9, 3, 0, 47, 25, 97324), datetime_complete=datetime.datetime(2026, 9, 3, 0, 47, 50, 99268), params={'channel_width': 16, 'num_blocks': 5, 'kernel_size': 5, 'dropout': 0.1, 'learning_rate': 0.002708160864249967, 'weight_decay': 0.0, 'batch_size': 64}, user_attrs={}, system_attrs={'completed_rung_0': 0.6009596884250641, 'completed_rung_1': 0.7423650324344635, 'completed_rung_2': 0.6748935977617899}, intermediate_values={1: 0.6009596884250641, 2: 0.7423650324344635, 4: 0.6748935977617899}, distributions={'channel_width': CategoricalDistribution(choices=(8, 16, 32)), 'num_blocks': CategoricalDistribution(choices=(5, 6, 7)), 'kernel_size': CategoricalDistribution(choices=(3, 5)), 'dropout': CategoricalDistribution(choices=(0.0, 0.1, 0.2)), 'learning_rate': FloatDistribution(high=0.003, log=True, low=0.0001, step=None), 'weight_decay': CategoricalDistribution(choices=(0.

In [ ]:
HYPERPARAM_DIR = "/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/data/IS/hyperparameters"

os.makedirs(HYPERPARAM_DIR, exist_ok=True)

with open(f"{HYPERPARAM_DIR}/tcn_study.pkl", "wb") as f:
    pickle.dump(tcn_study, f)

with open(f"{HYPERPARAM_DIR}/tcn_best_params.pkl", "wb") as f:
    pickle.dump(best_tcn_params, f)